# Purpose
This notebook will focus on feature engineering and data preprocessing techniques for machine learning models. We will explore various methods to clean, transform, and prepare the data for analysis, ensuring that it is in the best possible shape for model training and evaluation. (Broad)

# Imports

In [2]:
import pandas as pd 
import numpy as np
from pathlib import Path

# ENUMS

In [3]:
PROCESSED_DATA_PATH = Path("../data/processed")
INTEGRATED_DATA_PATH = PROCESSED_DATA_PATH / "integrated"
FEATURE_DATA_PATH = PROCESSED_DATA_PATH / "features"

## Create Feature Data Directory

In [5]:
if not FEATURE_DATA_PATH.exists():
    FEATURE_DATA_PATH.mkdir(parents=True, exist_ok=True)
else:
    print(f"Directory {FEATURE_DATA_PATH} already exists.")

Directory ..\data\processed\features already exists.


# Modeling Base CSV Setup

In [7]:
modeling_base = pd.read_csv(
    INTEGRATED_DATA_PATH / "modeling_base_dataset.csv",
    parse_dates=["Date"]
)

## Modeling Base Shape Validation

In [8]:
modeling_base.head()

,Date,ticker,adjusted_close,daily_return,risk_free_rate_pct,risk_free_rate_decimal,cpi_index,cpi_pct_change,company_name,gics_sector,asset_type
0,2018-01-03,AAPL,40.260067,-0.000174,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock
1,2018-01-04,AAPL,40.447075,0.004645,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock
2,2018-01-05,AAPL,40.907566,0.011385,1.39,0.0139,248.859,0.0,Apple Inc.,Information Technology,Stock
3,2018-01-08,AAPL,40.755634,-0.003714,1.45,0.0145,248.859,0.0,Apple Inc.,Information Technology,Stock
4,2018-01-09,AAPL,40.750969,-0.000114,1.44,0.0144,248.859,0.0,Apple Inc.,Information Technology,Stock


In [9]:
modeling_base.shape

(42189, 11)

In [10]:
modeling_base.info()

<class 'pandas.DataFrame'>
RangeIndex: 42189 entries, 0 to 42188
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Date                    42189 non-null  datetime64[us]
 1   ticker                  42189 non-null  str           
 2   adjusted_close          42189 non-null  float64       
 3   daily_return            42189 non-null  float64       
 4   risk_free_rate_pct      42189 non-null  float64       
 5   risk_free_rate_decimal  42189 non-null  float64       
 6   cpi_index               42189 non-null  float64       
 7   cpi_pct_change          42189 non-null  float64       
 8   company_name            42189 non-null  str           
 9   gics_sector             42189 non-null  str           
 10  asset_type              42189 non-null  str           
dtypes: datetime64[us](1), float64(6), str(4)
memory usage: 5.2 MB


In [11]:
modeling_base.isna().sum()

Date                      0
ticker                    0
adjusted_close            0
daily_return              0
risk_free_rate_pct        0
risk_free_rate_decimal    0
cpi_index                 0
cpi_pct_change            0
company_name              0
gics_sector               0
asset_type                0
dtype: int64

## Modeling Base Sorting 

Sorting Before Making Rolling Features

In [12]:
modeling_base = modeling_base.sort_values(['ticker','Date']).reset_index(drop=True)

In [15]:
modeling_base.head()

,Date,ticker,adjusted_close,daily_return,risk_free_rate_pct,risk_free_rate_decimal,cpi_index,cpi_pct_change,company_name,gics_sector,asset_type
0,2018-01-03,AAPL,40.260067,-0.000174,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock
1,2018-01-04,AAPL,40.447075,0.004645,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock
2,2018-01-05,AAPL,40.907566,0.011385,1.39,0.0139,248.859,0.0,Apple Inc.,Information Technology,Stock
3,2018-01-08,AAPL,40.755634,-0.003714,1.45,0.0145,248.859,0.0,Apple Inc.,Information Technology,Stock
4,2018-01-09,AAPL,40.750969,-0.000114,1.44,0.0144,248.859,0.0,Apple Inc.,Information Technology,Stock


## Modeling Base Lagged Returns Values

In [18]:
modeling_base["return_lag_1"] = modeling_base.groupby("ticker")["daily_return"].shift(1)

In [19]:
modeling_base["return_lag_5"] = modeling_base.groupby("ticker")["daily_return"].shift(5)

In [20]:
modeling_base.head()

,Date,ticker,adjusted_close,daily_return,risk_free_rate_pct,risk_free_rate_decimal,cpi_index,cpi_pct_change,company_name,gics_sector,asset_type,return_lag_1,return_lag_5
0,2018-01-03,AAPL,40.260067,-0.000174,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock,NaN,NaN
1,2018-01-04,AAPL,40.447075,0.004645,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock,-0.000174,NaN
2,2018-01-05,AAPL,40.907566,0.011385,1.39,0.0139,248.859,0.0,Apple Inc.,Information Technology,Stock,0.004645,NaN
3,2018-01-08,AAPL,40.755634,-0.003714,1.45,0.0145,248.859,0.0,Apple Inc.,Information Technology,Stock,0.011385,NaN
4,2018-01-09,AAPL,40.750969,-0.000114,1.44,0.0144,248.859,0.0,Apple Inc.,Information Technology,Stock,-0.003714,NaN


# Create Rolling Return Features

In [21]:
modeling_base['rolling_return_5d'] = (
    modeling_base.groupby('ticker')['daily_return']
    .rolling(window=5)
    .mean()
    .reset_index(level=0, drop=True)
)

In [22]:
modeling_base.head()

,Date,ticker,adjusted_close,daily_return,risk_free_rate_pct,risk_free_rate_decimal,cpi_index,cpi_pct_change,company_name,gics_sector,asset_type,return_lag_1,return_lag_5,rolling_return_5d
0,2018-01-03,AAPL,40.260067,-0.000174,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock,NaN,NaN,NaN
1,2018-01-04,AAPL,40.447075,0.004645,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock,-0.000174,NaN,NaN
2,2018-01-05,AAPL,40.907566,0.011385,1.39,0.0139,248.859,0.0,Apple Inc.,Information Technology,Stock,0.004645,NaN,NaN
3,2018-01-08,AAPL,40.755634,-0.003714,1.45,0.0145,248.859,0.0,Apple Inc.,Information Technology,Stock,0.011385,NaN,NaN
4,2018-01-09,AAPL,40.750969,-0.000114,1.44,0.0144,248.859,0.0,Apple Inc.,Information Technology,Stock,-0.003714,NaN,0.002405


In [23]:
modeling_base['rolling_return_20d'] = (
    modeling_base.groupby('ticker')['daily_return']
    .rolling(window=20)
    .mean()
    .reset_index(level=0, drop=True)
)

In [24]:
modeling_base.head()

,Date,ticker,adjusted_close,daily_return,risk_free_rate_pct,risk_free_rate_decimal,cpi_index,cpi_pct_change,company_name,gics_sector,asset_type,return_lag_1,return_lag_5,rolling_return_5d,rolling_return_20d
0,2018-01-03,AAPL,40.260067,-0.000174,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock,NaN,NaN,NaN,NaN
1,2018-01-04,AAPL,40.447075,0.004645,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock,-0.000174,NaN,NaN,NaN
2,2018-01-05,AAPL,40.907566,0.011385,1.39,0.0139,248.859,0.0,Apple Inc.,Information Technology,Stock,0.004645,NaN,NaN,NaN
3,2018-01-08,AAPL,40.755634,-0.003714,1.45,0.0145,248.859,0.0,Apple Inc.,Information Technology,Stock,0.011385,NaN,NaN,NaN
4,2018-01-09,AAPL,40.750969,-0.000114,1.44,0.0144,248.859,0.0,Apple Inc.,Information Technology,Stock,-0.003714,NaN,0.002405,NaN


# Create Rolling Volatility Feature

In [26]:
modeling_base['rolling_volatility_5d'] = (
    modeling_base.groupby('ticker')['daily_return']
    .rolling(window=5)
    .std()
    .reset_index(level=0, drop=True)
)

In [27]:
modeling_base.head()

,Date,ticker,adjusted_close,daily_return,risk_free_rate_pct,risk_free_rate_decimal,cpi_index,cpi_pct_change,company_name,gics_sector,asset_type,return_lag_1,return_lag_5,rolling_return_5d,rolling_return_20d,rolling_volatility_5d
0,2018-01-03,AAPL,40.260067,-0.000174,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock,NaN,NaN,NaN,NaN,NaN
1,2018-01-04,AAPL,40.447075,0.004645,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock,-0.000174,NaN,NaN,NaN,NaN
2,2018-01-05,AAPL,40.907566,0.011385,1.39,0.0139,248.859,0.0,Apple Inc.,Information Technology,Stock,0.004645,NaN,NaN,NaN,NaN
3,2018-01-08,AAPL,40.755634,-0.003714,1.45,0.0145,248.859,0.0,Apple Inc.,Information Technology,Stock,0.011385,NaN,NaN,NaN,NaN
4,2018-01-09,AAPL,40.750969,-0.000114,1.44,0.0144,248.859,0.0,Apple Inc.,Information Technology,Stock,-0.003714,NaN,0.002405,NaN,0.005833


In [29]:
modeling_base["rolling_volatility_20d"] = (
    modeling_base.groupby("ticker")["daily_return"]
    .rolling(window=20)
    .std()
    .reset_index(level=0, drop=True)
)

In [30]:
modeling_base.head()

,Date,ticker,adjusted_close,daily_return,risk_free_rate_pct,risk_free_rate_decimal,cpi_index,cpi_pct_change,company_name,gics_sector,asset_type,return_lag_1,return_lag_5,rolling_return_5d,rolling_return_20d,rolling_volatility_5d,rolling_volatility_20d
0,2018-01-03,AAPL,40.260067,-0.000174,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock,NaN,NaN,NaN,NaN,NaN,NaN
1,2018-01-04,AAPL,40.447075,0.004645,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock,-0.000174,NaN,NaN,NaN,NaN,NaN
2,2018-01-05,AAPL,40.907566,0.011385,1.39,0.0139,248.859,0.0,Apple Inc.,Information Technology,Stock,0.004645,NaN,NaN,NaN,NaN,NaN
3,2018-01-08,AAPL,40.755634,-0.003714,1.45,0.0145,248.859,0.0,Apple Inc.,Information Technology,Stock,0.011385,NaN,NaN,NaN,NaN,NaN
4,2018-01-09,AAPL,40.750969,-0.000114,1.44,0.0144,248.859,0.0,Apple Inc.,Information Technology,Stock,-0.003714,NaN,0.002405,NaN,0.005833,NaN


# Create Moving Average Feature

In [31]:
modeling_base['moving_avg_20d'] = (
    modeling_base.groupby('ticker')['adjusted_close']
    .rolling(window=20)
    .mean()
    .reset_index(level=0, drop=True)
)

In [32]:
modeling_base.head()

,Date,ticker,adjusted_close,daily_return,risk_free_rate_pct,risk_free_rate_decimal,cpi_index,cpi_pct_change,company_name,gics_sector,asset_type,return_lag_1,return_lag_5,rolling_return_5d,rolling_return_20d,rolling_volatility_5d,rolling_volatility_20d,moving_avg_20d
0,2018-01-03,AAPL,40.260067,-0.000174,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2018-01-04,AAPL,40.447075,0.004645,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock,-0.000174,NaN,NaN,NaN,NaN,NaN,NaN
2,2018-01-05,AAPL,40.907566,0.011385,1.39,0.0139,248.859,0.0,Apple Inc.,Information Technology,Stock,0.004645,NaN,NaN,NaN,NaN,NaN,NaN
3,2018-01-08,AAPL,40.755634,-0.003714,1.45,0.0145,248.859,0.0,Apple Inc.,Information Technology,Stock,0.011385,NaN,NaN,NaN,NaN,NaN,NaN
4,2018-01-09,AAPL,40.750969,-0.000114,1.44,0.0144,248.859,0.0,Apple Inc.,Information Technology,Stock,-0.003714,NaN,0.002405,NaN,0.005833,NaN,NaN


In [33]:
modeling_base["price_to_moving_avg_20d"] = (
    modeling_base["adjusted_close"] / modeling_base["moving_avg_20d"]
)

In [34]:
modeling_base.head()

,Date,ticker,adjusted_close,daily_return,risk_free_rate_pct,risk_free_rate_decimal,cpi_index,cpi_pct_change,company_name,gics_sector,asset_type,return_lag_1,return_lag_5,rolling_return_5d,rolling_return_20d,rolling_volatility_5d,rolling_volatility_20d,moving_avg_20d,price_to_moving_avg_20d
0,2018-01-03,AAPL,40.260067,-0.000174,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2018-01-04,AAPL,40.447075,0.004645,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock,-0.000174,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2018-01-05,AAPL,40.907566,0.011385,1.39,0.0139,248.859,0.0,Apple Inc.,Information Technology,Stock,0.004645,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2018-01-08,AAPL,40.755634,-0.003714,1.45,0.0145,248.859,0.0,Apple Inc.,Information Technology,Stock,0.011385,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2018-01-09,AAPL,40.750969,-0.000114,1.44,0.0144,248.859,0.0,Apple Inc.,Information Technology,Stock,-0.003714,NaN,0.002405,NaN,0.005833,NaN,NaN,NaN


# Create Future Volatility Target

In [35]:
modeling_base["future_volatility_20d"] = (
    modeling_base.groupby("ticker")["daily_return"]
    .rolling(window=20)
    .std()
    .shift(-20)
    .reset_index(level=0, drop=True)
)

In [36]:
modeling_base.head()

,Date,ticker,adjusted_close,daily_return,risk_free_rate_pct,risk_free_rate_decimal,cpi_index,cpi_pct_change,company_name,gics_sector,asset_type,return_lag_1,return_lag_5,rolling_return_5d,rolling_return_20d,rolling_volatility_5d,rolling_volatility_20d,moving_avg_20d,price_to_moving_avg_20d,future_volatility_20d
0,2018-01-03,AAPL,40.260067,-0.000174,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.009490
1,2018-01-04,AAPL,40.447075,0.004645,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock,-0.000174,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.013250
2,2018-01-05,AAPL,40.907566,0.011385,1.39,0.0139,248.859,0.0,Apple Inc.,Information Technology,Stock,0.004645,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.013567
3,2018-01-08,AAPL,40.755634,-0.003714,1.45,0.0145,248.859,0.0,Apple Inc.,Information Technology,Stock,0.011385,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.017207
4,2018-01-09,AAPL,40.750969,-0.000114,1.44,0.0144,248.859,0.0,Apple Inc.,Information Technology,Stock,-0.003714,NaN,0.002405,NaN,0.005833,NaN,NaN,NaN,0.017658


# Drop Rows That Cannot Be Used For Modeling

In [37]:
feature_dataset = modeling_base.dropna().copy()

In [38]:
feature_dataset.shape

(41390, 20)

# Check Final Dataset

In [39]:
feature_dataset.head()

,Date,ticker,adjusted_close,daily_return,risk_free_rate_pct,risk_free_rate_decimal,cpi_index,cpi_pct_change,company_name,gics_sector,asset_type,return_lag_1,return_lag_5,rolling_return_5d,rolling_return_20d,rolling_volatility_5d,rolling_volatility_20d,moving_avg_20d,price_to_moving_avg_20d,future_volatility_20d
19,2018-01-31,AAPL,39.138020,0.002755,1.46,0.0146,248.859,0.000000,Apple Inc.,Information Technology,Stock,-0.005894,-0.015929,-0.007870,-0.001378,0.011013,0.009462,40.695438,0.961730,0.022707
20,2018-02-01,AAPL,39.219841,0.002091,1.48,0.0148,249.529,0.002692,Apple Inc.,Information Technology,Stock,0.002755,-0.017851,-0.003882,-0.001265,0.010065,0.009490,40.643427,0.964974,0.022726
21,2018-02-02,AAPL,37.518093,-0.043390,1.48,0.0148,249.529,0.002692,Apple Inc.,Information Technology,Stock,0.002091,0.002338,-0.013027,-0.003667,0.019425,0.013250,40.496978,0.926442,0.019948
22,2018-02-05,AAPL,36.580715,-0.024985,1.51,0.0151,249.529,0.002692,Apple Inc.,Information Technology,Stock,-0.043390,-0.020698,-0.013885,-0.005485,0.019936,0.013567,40.280635,0.908146,0.018715
23,2018-02-06,AAPL,38.109501,0.041792,1.52,0.0152,249.529,0.002692,Apple Inc.,Information Technology,Stock,-0.024985,-0.005894,-0.004347,-0.003210,0.032292,0.017207,40.148329,0.949218,0.017050


In [40]:
feature_dataset.isna().sum()

Date                       0
ticker                     0
adjusted_close             0
daily_return               0
risk_free_rate_pct         0
risk_free_rate_decimal     0
cpi_index                  0
cpi_pct_change             0
company_name               0
gics_sector                0
asset_type                 0
return_lag_1               0
return_lag_5               0
rolling_return_5d          0
rolling_return_20d         0
rolling_volatility_5d      0
rolling_volatility_20d     0
moving_avg_20d             0
price_to_moving_avg_20d    0
future_volatility_20d      0
dtype: int64

# Save Final Notebook

In [41]:
feature_dataset.to_csv(
    FEATURE_DATA_PATH / "feature_engineered_dataset.csv",
    index=False
)